# 11 – Unseen-Vehicle Test Data for the RoadLens App

**Component 4 – Multi-Sensor ML Road Roughness Assessment and Cross-Vehicle-Driver Generalization**
Sandanayaka S D I K – IT23222472

The deployment model of Notebook 07 was trained on **all nine** PVS trips, so any PVS trip shown in the app
would be data the model has already seen. To demonstrate the app honestly, this notebook repeats one
**leave-one-vehicle-out (LOVO)** fold of the C7 smartphone configuration:

| | Vehicles | Drivers | Trips |
|---|---|---|---|
| Training | VW Saveiro, Fiat Bravo | Driver 1, Driver 2 | PVS 1–6 |
| **Held out (never seen)** | **Fiat Palio** | **Driver 3** | **PVS 7–9** |

Outputs (saved to `MyDrive/PVS_Model_Results_v2/app_test/` and downloaded as one zip):

1. `model_lovo_fiat_palio/` – the fold model in the portable format used by the app.
2. `PVS7/`, `PVS8/`, `PVS9/` – the held-out trips converted into the app's upload format
   (left dashboard accelerometer + gyroscope, 1 Hz GPS, sensor manifest) plus `labels.csv` for scoring.
3. `research_scores.csv` – the scores of this fold computed by the research pipeline, to compare with the
   app's results.

The training data, settings and code are exactly those of Notebooks 03–07 (`c4_pipeline.py`).

## 1. Setup

In [2]:
import os, sys, json, hashlib, importlib, shutil
import numpy as np
import pandas as pd
import sklearn, xgboost
from google.colab import drive
drive.mount('/content/drive')

CODE_DIR = '/content/drive/MyDrive/PVS_Code'
sys.path.insert(0, CODE_DIR)
import c4_pipeline as c4
importlib.reload(c4)
c4.check_setup()

HELD_OUT = 'Fiat Palio'
OUT_DIR = os.path.join(c4.RESULTS_DIR, 'app_test')
MODEL_DIR = os.path.join(OUT_DIR, 'model_lovo_fiat_palio')
os.makedirs(MODEL_DIR, exist_ok=True)
print('Output folder:', OUT_DIR)

Mounted at /content/drive
Drive folder      : /content/drive/MyDrive
Synced PVS1 found : True
Processed folder  : /content/drive/MyDrive/PVS_Processed True
Feature folder    : /content/drive/MyDrive/PVS_Feature_Datasets_v2 True
Results folder    : /content/drive/MyDrive/PVS_Model_Results_v2 True
Output folder: /content/drive/MyDrive/PVS_Model_Results_v2/app_test


## 2. Train the C7 model without the held-out vehicle

Same configuration as the deployment model (C7: left dashboard sensor, four invariant channels, speed
normalisation, per-trip normalisation, manoeuvre filter, cross-vehicle stable feature selection, XGBoost).
Speed slopes and feature selection are learned **only from the training trips**.

In [3]:
data = c4.load_data()
CFG7 = c4.ablation_configs()['C7 Deployment (phone)']
vehicle = data.A.vehicle.values

train_mask = vehicle != HELD_OUT
tr = c4.train_index(data, CFG7, train_mask)
X, info = c4.prepare(data, CFG7, tr)
model = c4.fit('XGB', X.values[tr], data.y[tr])

print('Training windows :', len(tr), '| trips', sorted(set(data.A.pvs.values[tr].astype(int).tolist())))
print('Selected features:', X.shape[1])
assert HELD_OUT not in set(vehicle[tr]), 'Held-out vehicle leaked into training'

Training windows : 5635 | trips [1, 2, 3, 4, 5, 6]
Selected features: 40


## 3. Research-pipeline score on the held-out vehicle

The held-out windows are scored exactly as in Notebook 07 (evaluation set **E**: labelled, moving, no speed
bump, no manoeuvre). This is the reference the app result is compared with.

In [4]:
rows = []
te_all = np.where((vehicle == HELD_OUT) & data.E)[0]
for name, idx in [('PVS 7-9 (all)', te_all)] + [(f'PVS{n}', te_all[data.A.pvs.values[te_all] == n]) for n in (7, 8, 9)]:
    p = model.predict(X.values[idx])
    rows.append({'test': name, 'windows': len(idx), **c4.metrics(data.y[idx], p)})
research = pd.DataFrame(rows)
research.to_csv(os.path.join(OUT_DIR, 'research_scores.csv'), index=False)
research.round(3)

,test,windows,accuracy,precision,recall,macro_f1,f1_good,f1_regular,f1_bad,recall_bad
0,PVS 7-9 (all),2644,0.861,0.828,0.843,0.833,0.990,0.828,0.682,0.746
1,PVS7,921,0.870,0.748,0.774,0.758,0.986,0.883,0.404,0.475
2,PVS8,991,0.900,0.893,0.900,0.890,0.990,0.822,0.858,0.781
3,PVS9,732,0.796,0.755,0.821,0.731,0.995,0.731,0.465,0.877


## 4. Export the fold model to the portable format

As in Notebook 10: imputer medians, scaler parameters (with the scaler's exact float32 rounding) and the
XGBoost trees, checked against the scikit-learn pipeline.

In [5]:
def plain(o):
    if isinstance(o, dict):
        return {str(k): plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [plain(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o

steps = dict(model.named_steps)
imp, sc, clf = steps['simpleimputer'], steps['standardscaler'], steps['xgbclassifier']
nf = X.shape[1]
Xs = np.random.default_rng(1).standard_normal((500, nf)).astype(np.float32)
ref = sc.transform(imp.transform(Xs))
per_step = Xs.copy(); per_step -= sc.mean_; per_step /= sc.scale_
f32p = (Xs - sc.mean_.astype(np.float32)) / sc.scale_.astype(np.float32)
scale_arithmetic = 'float64_per_step' if np.array_equal(per_step, ref) else 'float32_params' if np.array_equal(f32p, ref) else None
assert scale_arithmetic

booster_file = 'c4_model_portable.xgb.json'
clf.get_booster().save_model(os.path.join(MODEL_DIR, booster_file))
f1 = float(research.iloc[0]['macro_f1'])
meta = plain({
    'model_name': 'XGB', 'model_version': f'XGB-C7-LOVO-heldout-FiatPalio-f1{f1:.3f}', 'demo': False,
    'features': list(X.columns), 'speed_slopes': info.get('slopes'), 'speed_ref_log': info.get('ref'),
    'config': dict(c4.CFG), 'sensor': c4.CFG['DEPLOY_SENSOR'], 'channels': c4.ROBUST_CH, 'classes': c4.CLASSES,
    'lovo_macro_f1': f1, 'held_out_vehicle': HELD_OUT, 'training_trips': [1, 2, 3, 4, 5, 6],
    'notes': 'LOVO fold model for an honest app demonstration: Fiat Palio / Driver 3 (PVS 7-9) never seen in training.',
    'booster_file': booster_file, 'impute_median': imp.statistics_, 'scale_mean': sc.mean_, 'scale_std': sc.scale_,
    'scale_arithmetic': scale_arithmetic,
    'exported_with': {'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__}})
with open(os.path.join(MODEL_DIR, 'c4_model_portable.json'), 'w') as f:
    json.dump(meta, f, indent=1, allow_nan=False)

# check: portable arithmetic == scikit-learn pipeline
def portable_predict(Xq):
    b = xgboost.Booster(); b.load_model(os.path.join(MODEL_DIR, booster_file))
    Xq = np.where(np.isfinite(Xq), Xq, imp.statistics_.astype(np.float32)).astype(np.float32)
    if scale_arithmetic == 'float64_per_step':
        Xq = (Xq - sc.mean_).astype(np.float32); Xq = (Xq / sc.scale_).astype(np.float32)
    else:
        Xq = (Xq - sc.mean_.astype(np.float32)) / sc.scale_.astype(np.float32)
    return b.predict(xgboost.DMatrix(Xq))
Xq = X.values[te_all].astype(np.float32)
diff = float(np.abs(portable_predict(Xq) - model.predict_proba(Xq)).max())
print('Model version :', meta['model_version'])
print('Scaler        :', scale_arithmetic)
print(f'Max probability difference (held-out windows): {diff:.2e}')
assert diff < 1e-5

Model version : XGB-C7-LOVO-heldout-FiatPalio-f10.833
Scaler        : float64_per_step
Max probability difference (held-out windows): 0.00e+00


## 5. Convert the held-out trips into the app's upload format

Only the **left dashboard** sensor is used (the smartphone-like position of the deployed model). Timestamps
become integer milliseconds from the trip start; GPS is reduced to one fix per second, as a phone records it.
`labels.csv` (worst-side road condition per sample) is for scoring only and is not uploaded.

In [6]:
def to_contract(n):
    df = pd.read_csv(c4.synced_path(n))
    df = df.sort_values('timestamp').drop_duplicates('timestamp').reset_index(drop=True)
    units = c4.detect_units(df)
    df = c4.harmonise_units(df, units)
    ms = np.round((df.timestamp - df.timestamp.iat[0]) * 1000).astype('int64')
    acc, gyr = c4.sensor_cols('left_dashboard', 'acc'), c4.sensor_cols('left_dashboard', 'gyro')
    imu = pd.DataFrame({'timestamp_ms': ms})
    for src, dst in zip(acc + gyr, ['ax_mps2', 'ay_mps2', 'az_mps2', 'gx_rads', 'gy_rads', 'gz_rads']):
        imu[dst] = df[src].round(5)
    first = ~(ms // 1000).duplicated()
    gps = pd.DataFrame({'timestamp_ms': ms[first], 'latitude': df.latitude[first].round(7),
                        'longitude': df.longitude[first].round(7), 'accuracy_m': np.nan,
                        'speed_mps': df.speed[first].round(3)})
    d = os.path.join(OUT_DIR, f'PVS{n}')
    os.makedirs(d, exist_ok=True)
    imu.dropna().to_csv(os.path.join(d, 'imu.csv'), index=False)
    gps.to_csv(os.path.join(d, 'gps.csv'), index=False)
    pd.DataFrame({'timestamp_ms': ms, 'road_condition': df['road_condition']}).to_csv(os.path.join(d, 'labels.csv'), index=False)
    vehicle_name, driver, route = c4.PVS_META[n]
    with open(os.path.join(d, 'manifest.json'), 'w') as f:
        json.dump({'schema_version': '1.0', 'device': f'PVS{n} MPU-9250 left dashboard ({vehicle_name}, {driver})',
                   'axis_frame': 'device', 'gravity_included': True, 'sample_rate_hz': 100, 'gps_rate_hz': 1,
                   'clock_origin': 'capture_start', 'source': 'PVS dataset', 'route': int(route)}, f, indent=2)
    return {'trip': f'PVS{n}', 'vehicle': vehicle_name, 'driver': driver, 'route': route,
            'minutes': round((ms.iat[-1]) / 60000, 1), 'imu_rows': len(imu), 'gps_fixes': int(first.sum()),
            'imu_MB': round(os.path.getsize(os.path.join(d, 'imu.csv')) / 1e6, 1)}

trips = pd.DataFrame([to_contract(n) for n in (7, 8, 9)])
trips

,trip,vehicle,driver,route,minutes,imu_rows,gps_fixes,imu_MB
0,PVS7,Fiat Palio,Driver 3,1,21.4,128548,1286,7.4
1,PVS8,Fiat Palio,Driver 3,2,20.6,123618,1237,7.1
2,PVS9,Fiat Palio,Driver 3,3,15.3,91555,916,5.2


## 6. Download everything as one zip

In [7]:
zip_path = shutil.make_archive('/content/c4_app_test_fiat_palio', 'zip', OUT_DIR)
print(f'{zip_path}  {os.path.getsize(zip_path) / 1e6:.1f} MB')
from google.colab import files
files.download(zip_path)

/content/c4_app_test_fiat_palio.zip  8.7 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Use in the app

1. Unzip. Copy `model_lovo_fiat_palio/` to `RoadLens/ml/c4/models/production/` (replacing the files there
   while presenting this demo; keep the all-trip model files in a separate folder to switch back).
2. `docker compose up --build`, open `http://localhost:3000/c4`. The badge shows
   *XGB-C7-LOVO-heldout-FiatPalio-…*.
3. Upload `PVS7/imu.csv`, `PVS7/gps.csv` and `PVS7/manifest.json`. The map shows the road condition of a
   vehicle and driver the model has never seen.
4. Compare with the labels using `ml/c4/scripts/check_on_pvs.py` (uses `labels.csv`) and with
   `research_scores.csv`.